In [ ]:
import pandas as pd
from pathlib import Path

# Buat Claim Dataset Ujaran Kebencian

## Bersihin Artikel

In [ ]:
folderData = Path("Dataset_UjaranKebencian/artikel")
files = []


for file in folderData.glob("*.jsonl"):
    df_sementara = pd.read_json(file, lines=True)
    files.append(df_sementara)

df_artikel = pd.concat(files, ignore_index=True)

In [ ]:
df_artikel.isna().sum()

In [ ]:
df_artikel = df_artikel.drop(df_artikel[df_artikel["$"].notna()].index)

In [ ]:
df_artikel.columns.unique()

In [ ]:
df_artikel = df_artikel[["judul", "teks", "target", "label"]]

In [ ]:
df_artikel.isna().sum()

In [ ]:
def bersihinjudul(df):
    target = str(df["judul"])
    judul = target.split(':')

    return " ".join(judul[1:]).strip()

df_artikel["judul"] = df_artikel.apply(bersihinjudul, axis=1)

In [ ]:
mapping = {
    # ======================
    # SUKU / ETNIS
    # ======================
    "Suku Bali": "suku_bali",

    "Suku Batak": "suku_batak",

    "Suku Bugis": "suku_bugis",

    "Suku Minang": "suku_minang",
    "Perempuan Minang": "suku_minang",
    "Etnis Minang": "suku_minang",
    "Orang Minang": "suku_minang",

    "Suku Dayak": "suku_dayak",

    "Suku Jawa": "suku_jawa",
    "Orang Jawa": "suku_jawa",
    "Etnis Jawa": "suku_jawa",
    "Masyarakat Jawa": "suku_jawa",

    "Suku Madura": "suku_madura",
    "Etnis Madura": "suku_madura",
    "Orang Madura": "suku_madura",
    "Pedagang Madura": "suku_madura",
    "Warga Madura": "suku_madura",
    "Masyarakat Madura": "suku_madura",

    "Suku Melayu": "suku_melayu",
    "Masyarakat Melayu": "suku_melayu",
    "Orang Melayu": "suku_melayu",

    "Suku Papua": "suku_papua",

    "Suku Sunda": "suku_sunda",
    "Masyarakat Sunda": "suku_sunda",
    "Orang Tua Sunda": "suku_sunda",
    "Komunitas Sunda": "suku_sunda",
    "Laki-laki Sunda": "suku_sunda",
    "Perempuan Sunda": "suku_sunda",
    "Pria pedesaan Jawa Barat": "suku_sunda",
    "Lelaki Jawa Barat": "suku_sunda",
    "Masyarakat Jawa Barat": "suku_sunda",
    "Laki-laki di pedesaan Jawa Barat": "suku_sunda",
    "Pengangguran di Jawa Barat": "suku_sunda",
    "Laki-laki di Tanah Sunda": "suku_sunda",

    "Suku Tionghoa": "suku_tionghoa",
    "Suku Tionghoa atau Cina": "suku_tionghoa",

    # ======================
    # AGAMA
    # ======================
    "Penganut Islam": "agama_islam",
    "Penganut Kristen": "agama_kristen",

    # ======================
    # GENDER
    # ======================
    "Laki-laki Indonesia": "gender_laki_laki",
    "Perempuan Indonesia": "gender_perempuan",

    # ======================
    # KOMUNITAS / LAINNYA
    # ======================
    "PSHT": "komunitas_psht",
    "Komunitas PSHT": "komunitas_psht",
    "Komunitas Silat Jawa (PSHT)": "komunitas_psht",

    "Pengepul": "pekerjaan_pengepul",
    "Pengepul Madura": "pekerjaan_pengepul",
    "Pengepul Rongsokan": "pekerjaan_pengepul",

    "Orang Tua": "kelompok_orang_tua",
    "Orang Tua Indonesia": "kelompok_orang_tua",
    "Orang Tua yang Mengirim Anak ke Pesantren": "kelompok_orang_tua",

    "Masyarakat yang sering ngeteh dan mengobrol berjam-jam di kedai kopi": "kebiasaan_ngopi",
    "Masyarakat yang gemar 'ngeteh' dan ngobrol di kedai kopi": "kebiasaan_ngopi",
}

df_artikel["target"] = df_artikel["target"].map(mapping).fillna("unknown")

In [ ]:
df_artikel[df_artikel["target"] == "komunitas_psht"].head(2)

In [ ]:
df_artikel = df_artikel.to_dict("records")

### ngerapihin format punya Artikel

In [ ]:
def artikelmaker(df):
    data = []
    jenis = "artikel"

    for baris in df:
        judul = baris.get("judul", "")
        artikel = baris.get("teks", "")
        target =  baris.get("target", "")
        label =  baris.get("label", "")
        

        fullArtikel = (
        f"Judul: {judul}.\n\n"
        f"Teks: {artikel}"
        )

        artikelAsli = {
            "claim_disinformasi": fullArtikel,
            "target": target,
            "label": label,
            "jenis": jenis
        }

        data.append(artikelAsli)

    return data

df_uk_artikel = artikelmaker(df_artikel)
        

## Bersihin Komentar

In [ ]:
folderData = Path("Dataset_UjaranKebencian/komentar")
komentar = []


for file in folderData.glob("*.jsonl"):
    df_sementara = pd.read_json(file, lines=True)
    komentar.append(df_sementara)

df_komentar = pd.concat(komentar, ignore_index=True)

In [ ]:
df_komentar

### ngerapihin format punya Komentar

In [ ]:
df_komentar = df_komentar[["komentar", "target", "label"]]
df_komentar = df_komentar.rename(columns={"komentar": "claim_disinformasi"})
df_komentar["jenis"] = "komentar"

In [ ]:
df_komentar.head(2)

In [ ]:
df_komentar = df_komentar.to_dict("records")

### Gabungin kedua dataset

In [ ]:
df_ujarankebencian = df_uk_artikel + df_komentar
df_ujarankebencian = pd.DataFrame(df_ujarankebencian)
df_ujarankebencian["claim_disinformasi"]  = df_ujarankebencian["claim_disinformasi"].apply(lambda x: str(x).encode().decode('unicode_escape')) 
df_ujarankebencian = df_ujarankebencian[~df_ujarankebencian["claim_disinformasi"].str.contains(r'[^\x00-\x7F]', na=False)] 

df_ujarankebencian.to_json("ujaranKebencian.jsonl", orient="records", lines=True)

In [ ]:
df_ujarankebencian